# 05_Global_Adoption_GoogleTrends
Build Global Adoption Score using Google Trends and merge into SES dataset.

In [2]:
!pip install pytrends -q

In [3]:
import pandas as pd
import numpy as np
import time
from pytrends.request import TrendReq
from sklearn.preprocessing import MinMaxScaler

In [4]:
master = pd.read_csv('master_skill_dataset_v3.csv')
master.head()

,skill,linkedin_demand,salary_premium,geographic_spread,current_usage,future_interest,future_score
0,aws,0.502949,0.571429,1.0,0.000000,0.000000,0.311452
1,azure,0.288271,0.497905,1.0,0.000000,0.000000,0.246753
2,c#,0.188676,0.462857,1.0,0.435240,0.492004,0.454085
3,c++,0.265800,0.837857,1.0,0.368799,0.414020,0.492355
4,django,0.015208,0.527619,1.0,0.155633,0.189361,0.272566


In [5]:
TECH_SKILLS = master['skill'].dropna().unique().tolist()
print(len(TECH_SKILLS))

35


In [6]:
pytrends = TrendReq(hl='en-US', tz=360)

In [7]:
rows=[]
for skill in TECH_SKILLS:
    try:
        pytrends.build_payload([skill], timeframe='today 5-y')
        geo = pytrends.interest_by_region(resolution='COUNTRY',inc_low_vol=True,inc_geo_code=False)
        if len(geo)==0:
            rows.append([skill,0,0])
            continue
        rows.append([skill,len(geo),geo[skill].sum()])
        time.sleep(2)
    except:
        rows.append([skill,0,0])

In [8]:
global_adoption = pd.DataFrame(rows,columns=['skill','countries_present','raw_global_interest'])

In [9]:
global_adoption['global_adoption_score']=MinMaxScaler().fit_transform(global_adoption[['raw_global_interest']])

In [10]:
master_v4 = master.merge(global_adoption[['skill','global_adoption_score']],on='skill',how='left')

In [11]:
df = pd.read_csv("master_skill_dataset_v4.csv")

df.sort_values(
    "future_score",
    ascending=False
)[
    [
        "skill",
        "linkedin_demand",
        "salary_premium",
        "current_usage",
        "future_interest",
        "global_adoption_score"
    ]
].head(20)

,skill,linkedin_demand,salary_premium,current_usage,future_interest,global_adoption_score
23,python,1.000000,0.523810,0.819348,0.953735,0.119525
31,sql,0.870150,0.379048,0.818361,0.852943,0.282513
11,javascript,0.375466,0.428571,1.000000,0.905262,0.627278
5,docker,0.248425,0.571429,0.779340,1.000000,0.073005
21,postgresql,0.064163,0.602619,0.681105,0.914058,0.225467
34,typescript,0.127122,0.577143,0.617465,0.770657,0.146350
10,java,0.551142,0.539000,0.486477,0.406214,0.176344
3,c++,0.265800,0.837857,0.368799,0.414020,1.000000
28,rust,0.022431,0.809524,0.201616,0.656157,0.313413
18,node.js,0.091008,0.582095,0.527366,0.561077,0.121902


In [12]:
global_adoption.to_csv('global_adoption.csv',index=False)
master_v4.to_csv('master_skill_dataset_v4.csv',index=False)
print('Saved')

Saved
